# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadhany222/flyrank-ml-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal check 1 — staleness (behind the refresh flags).** FlyRank's `stale_visible_page`
reason code fires on `days_since_last_update >= 180` AND `impressions_90d >= 500`. I'm checking
whether staleness (via `freshness_tier`) actually associates with decline before I lean on it.

**Signal check 2 — CTR vs position (behind the CTR-fix logic).** FlyRank's `low_ctr_visible_page`
reason code fires on visible pages with `avg_position` between 1-20 and `ctr < 0.5`. I'm checking
whether CTR really does behave differently across position tiers, since a low-CTR flag is only
meaningful if CTR should be higher at that position.

**The rule, in plain words:** A page is worth reviewing if it's stale, it's visible enough to
matter, and it's already declining — three signals combining because any one alone can be noise
(see w03: adding a rule this narrow only ever catches 0.2% of visible pages).

**Reason code:** `stale_declining_visible` — the one reason code this rule can output.

**Action label:** `review_for_refresh`

In [1]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadhany222/flyrank-ml-assignment1"
REPO_DIR = "flyrank-ml-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- SIGNAL CHECK 1: staleness (freshness_tier) vs decline rate ---
visible = df[df["impressions_90d"] >= 100].copy()
visible["is_declining"] = (visible["trend_direction"] == "down").astype(int)

signal1 = visible.groupby("freshness_tier").agg(
    n=("is_declining", "size"),
    decline_rate=("is_declining", "mean")
).round(3)
print("SIGNAL 1: freshness_tier vs decline rate (visible pages only)")
print(signal1)
print()

# --- SIGNAL CHECK 2: CTR by position_tier ---
signal2 = visible.groupby("position_tier").agg(
    n=("ctr", "size"),
    mean_ctr=("ctr", "mean")
).round(3)
print("SIGNAL 2: position_tier vs mean CTR (visible pages only)")
print(signal2)

SIGNAL 1: freshness_tier vs decline rate (visible pages only)
                    n  decline_rate
freshness_tier                     
0-30            13735         0.583
181+               35         0.743
31-90             152         0.592
91-180           8084         0.622

SIGNAL 2: position_tier vs mean CTR (visible pages only)
                  n  mean_ctr
position_tier                
deep            879     0.055
page_1         8633     0.355
page_3_5       6058     0.142
striking       5903     0.256
top_3           533     0.334


**Verdict, signal 1 (staleness → decline): MIXED.** Decline rate does rise with staleness
(0.583 at 0-30 days since update, up to 0.743 at 181+ days), but the trend is weak in the
middle (0.592 → 0.622) and the strongest bucket (`181+`) only has **n=35** — too small to
trust on its own. Staleness alone is a soft signal, not a strong one; it's worth keeping in
the rule only combined with other conditions, which is exactly what my rule does.

**Verdict, signal 2 (position → CTR): CONFIRMED.** Mean CTR rises cleanly and monotonically as
position improves: deep=0.055 (n=879) → page_3_5=0.142 (n=6,058) → striking=0.256 (n=5,903) →
top_3=0.334 (n=533) → page_1=0.355 (n=8,633). This is the real signal behind FlyRank's CTR-fix
logic, and it holds clearly in this data — a low CTR at a good position really is unusual, not noise.

## 2. Build the ranked queue (writes the CSV)

Score = stale × visible × declining, using only current-window signals — no future-window or
label-derived columns beyond `trend_direction` itself (which is the proxy this whole lane already
uses, same as w01–w03, explicitly not a model feature elsewhere).

In [2]:
import os

os.makedirs("work/outputs", exist_ok=True)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 100).astype(int)
declining = (df["trend_direction"] == "down").astype(int)

df["baseline_score"] = stale * visible_flag * declining * df["impressions_90d"]
df["reason_code"] = "stale_declining_visible"
df["action"] = df["baseline_score"].apply(lambda s: "review_for_refresh" if s > 0 else "monitor")

queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)

out_path = "work/outputs/baseline_action_score.csv"
queue[["content_id", "client_id", "baseline_score", "reason_code", "action",
       "impressions_90d", "days_since_last_update", "trend_direction"]].to_csv(out_path, index=False)

n_flagged = (queue["baseline_score"] > 0).sum()
print(f"Wrote {out_path}")
print(f"Total rows: {len(queue)}, flagged for review: {n_flagged} ({n_flagged/len(queue):.1%})")
queue.head(10)[["content_id", "baseline_score", "reason_code", "action", "impressions_90d", "trend_direction"]]

Wrote work/outputs/baseline_action_score.csv
Total rows: 30000, flagged for review: 26 (0.1%)


,content_id,baseline_score,reason_code,action,impressions_90d,trend_direction
0,content_cf56e2e2e282,61678,stale_declining_visible,review_for_refresh,61678,down
1,content_7368877ea310,59472,stale_declining_visible,review_for_refresh,59472,down
2,content_1bfaa38ff26c,25715,stale_declining_visible,review_for_refresh,25715,down
3,content_0a91db491d14,13299,stale_declining_visible,review_for_refresh,13299,down
4,content_5feee3994adb,7812,stale_declining_visible,review_for_refresh,7812,down
5,content_c2d929d83eaa,7558,stale_declining_visible,review_for_refresh,7558,down
6,content_b16bd7307b39,4590,stale_declining_visible,review_for_refresh,4590,down
7,content_fe16a55cd13d,4556,stale_declining_visible,review_for_refresh,4556,down
8,content_ecb6215e79fd,4429,stale_declining_visible,review_for_refresh,4429,down
9,content_928af3e22c80,1697,stale_declining_visible,review_for_refresh,1697,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top20 = queue.head(20)[["content_id", "action", "reason_code", "baseline_score",
                          "impressions_90d", "days_since_last_update", "avg_position", "ctr"]]
print(top20.to_string())
top20

              content_id              action              reason_code  baseline_score  impressions_90d  days_since_last_update  avg_position   ctr
0   content_cf56e2e2e282  review_for_refresh  stale_declining_visible           61678            61678                     194          19.7  0.15
1   content_7368877ea310  review_for_refresh  stale_declining_visible           59472            59472                     194          24.8  0.13
2   content_1bfaa38ff26c  review_for_refresh  stale_declining_visible           25715            25715                     194          22.2  0.23
3   content_0a91db491d14  review_for_refresh  stale_declining_visible           13299            13299                     193          10.5  0.49
4   content_5feee3994adb  review_for_refresh  stale_declining_visible            7812             7812                     194          39.0  0.01
5   content_c2d929d83eaa  review_for_refresh  stale_declining_visible            7558             7558                

,content_id,action,reason_code,baseline_score,impressions_90d,days_since_last_update,avg_position,ctr
0,content_cf56e2e2e282,review_for_refresh,stale_declining_visible,61678,61678,194,19.7,0.15
1,content_7368877ea310,review_for_refresh,stale_declining_visible,59472,59472,194,24.8,0.13
2,content_1bfaa38ff26c,review_for_refresh,stale_declining_visible,25715,25715,194,22.2,0.23
3,content_0a91db491d14,review_for_refresh,stale_declining_visible,13299,13299,193,10.5,0.49
4,content_5feee3994adb,review_for_refresh,stale_declining_visible,7812,7812,194,39.0,0.01
5,content_c2d929d83eaa,review_for_refresh,stale_declining_visible,7558,7558,193,17.9,0.20
6,content_b16bd7307b39,review_for_refresh,stale_declining_visible,4590,4590,194,31.0,0.00
7,content_fe16a55cd13d,review_for_refresh,stale_declining_visible,4556,4556,194,16.4,0.33
8,content_ecb6215e79fd,review_for_refresh,stale_declining_visible,4429,4429,194,25.3,0.38
9,content_928af3e22c80,review_for_refresh,stale_declining_visible,1697,1697,193,15.8,0.12


For each of the top 20 — action, why it's there, what would make it wrong:

1. **content_cf56e2e2e282** (score 61,678) → `review_for_refresh`. Very high impressions,
   stale (194 days), position 19.7, CTR 0.15. Would be wrong if: the decline is seasonal, not
   a real content problem.
2. **content_7368877ea310** (score 59,472) → `review_for_refresh`. Similarly high impressions,
   stale (194 days), position 24.8, CTR 0.13. Would be wrong if: a sibling page absorbed this
   page's traffic (consolidation, not decay).
3. **content_1bfaa38ff26c** (score 25,715) → `review_for_refresh`. Strong impressions, stale
   (194 days), position 22.2, CTR 0.23. Would be wrong if: this is a known low-priority page
   the client doesn't actually care about.
4. **content_0a91db491d14** (score 13,299) → `review_for_refresh`. Best position of the top 4
   (10.5), decent CTR (0.49) — a real page-one-decay-risk case. Would be wrong if: the position
   is about to improve on its own (temporary SERP volatility).
5. **content_5feee3994adb** (score 7,812) → `review_for_refresh`. Weak position (39.0) and very
   low CTR (0.01). Would be wrong if: this page never ranked well to begin with — nothing to "fix."
6. **content_c2d929d83eaa** (score 7,558) → `review_for_refresh`. Position 17.9, CTR 0.20. Would
   be wrong if: recent content changes already addressed the decline (stale
   `days_since_last_update` might not reflect a recent unlogged edit).
7. **content_b16bd7307b39** (score 4,590) → `review_for_refresh`. Position 31.0, CTR 0.00 —
   zero clicks despite thousands of impressions. Would be wrong if: the topic itself has
   permanently lower demand now (real-world relevance drop, not a fixable content issue).
8. **content_fe16a55cd13d** (score 4,556) → `review_for_refresh`. Position 16.4, CTR 0.33 —
   reasonably healthy CTR for a "declining" page. Would be wrong if: this page duplicates
   another higher-performing page on the same site (merge candidate, not refresh candidate).
9. **content_ecb6215e79fd** (score 4,429) → `review_for_refresh`. Position 25.3, CTR 0.38. Would
   be wrong if: impressions this low relative to the top rows mean the "decline" here is just
   noise on a small base.
10. **content_928af3e22c80** (score 1,697) → `review_for_refresh`. Position 15.8, CTR 0.12, but
    score much lower than rows 1-9. Would be wrong if: 1,697 impressions is too small a base to
    call a real decline rather than natural variance.
11. **content_e3ff1b093148** (score 1,408) → `review_for_refresh`. Only 183 days since update —
    right at the threshold — but strong position (7.8) and CTR (0.28). Would be wrong if: the
    page barely crosses the staleness threshold and isn't meaningfully "old" in practice.
12. **content_7f116ae1f6f5** (score 954) → `review_for_refresh`. 301 days stale (genuinely old),
    strong position (9.0), highest CTR yet (0.42). Would be wrong if: strong CTR/position mean
    this page is actually fine and just has a temporarily lower impression count.
13. **content_77d4d5930e5e** (score 828) → `review_for_refresh`. Position 18.6, CTR 0.24. Would
    be wrong if: the low score itself signals this page barely matters — not worth reviewer time
    compared to the top 10.
14. **content_72496874f806** (score 821) → `review_for_refresh`. 301 days stale, strong position
    (5.8), CTR 0.24. Would be wrong if: this page is stable at a low but consistent traffic
    level, not actually "declining" in a way worth acting on.
15. **content_6226ee6adc91** (score 545) → `review_for_refresh`. Position 17.8, CTR 0.18, small
    base (545 impressions). Would be wrong if: this is simply too small a page to justify
    reviewer time given limited capacity.
16. **content_074ba6ead17b** (score 533) → `review_for_refresh`. Weak position (48.0), CTR 0.00.
    Would be wrong if: a page ranking this poorly was never going to get clicks regardless of
    content quality — a refresh wouldn't fix a ranking problem this bad.
17. **content_fd16e3475c29** (score 429) → `review_for_refresh`. Position 9.0 (good), CTR 0.00 —
    a real anomaly (good position, zero clicks). Would be wrong if: this is a tracking/attribution
    gap rather than an actual CTR problem.
18. **content_ba00ffc6318c** (score 345) → `review_for_refresh`. Position 4.9 (excellent), but
    CTR **4.93** — far higher than every other row (all under 0.5). This looks like a data
    anomaly, not a genuine decline case; flagged below as the clearest weak pick in the top 20.
19. **content_4f241bad48a3** (score 285) → `review_for_refresh`. Position 19.1, CTR 0.00. Would
    be wrong if: this page's low score means it's below the threshold where reviewer time is
    worth spending at all.
20. **content_ea41fe5cf292** (score 265) → `review_for_refresh`. Position 8.0 (good), CTR 0.00 —
    another good-position/zero-CTR anomaly. Would be wrong if: this is the same tracking gap
    pattern as row 17, not a real content issue.

**Weak pick:** Row 18 (`content_ba00ffc6318c`, score 345) — checked the raw numbers
(17 clicks / 345 impressions = 4.93% CTR, confirmed correct, not a bug). This page has an
excellent position (4.9) and a CTR roughly 10x higher than every other row in the top 20
(all between 0.00-0.49%). A page performing this well on every visible signal still getting
flagged as "declining and stale" is suspicious — it suggests either a very recent, sharp drop
from an even higher baseline (real decline, worth reviewing) or that `trend_direction` is
being driven by something my rule doesn't see (e.g. seasonal spike in the prior 30-day window
that's now normalizing, not a genuine problem). This is exactly the kind of case a reviewer
should look at directly rather than trust the rule blindly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.